# Experiment: codex
## 0 Setup
Prepare a reproducible workspace, confirm required local inputs, and read the variable dictionary (without loading the data file yet).

In [1]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import ParameterGrid, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBClassifier

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

BASE_DIR = Path('.')
DATA_DIR = BASE_DIR / 'data'
EVIDENCE_DIR = BASE_DIR / 'evidence_codex'
EDA_PICS_DIR = EVIDENCE_DIR / 'EDA_codex_Pics'

DATA_DICT_PATH = DATA_DIR / 'participation_2024-25_data_dictionary_cleaned.txt'
DATA_FILE_PATH = DATA_DIR / 'participation_2024-25_experiment.tab'

EVIDENCE_DIR.mkdir(exist_ok=True)
EDA_PICS_DIR.mkdir(exist_ok=True)

assert DATA_DICT_PATH.exists(), f'Missing input dictionary: {DATA_DICT_PATH}'
assert DATA_FILE_PATH.exists(), f'Missing input data file: {DATA_FILE_PATH}'

with DATA_DICT_PATH.open('r', encoding='utf-8') as f:
    dictionary_preview = ''.join(f.readlines()[:20])

print('Global random seed set to:', RANDOM_STATE)
print('Input dictionary located at:', DATA_DICT_PATH)
print('Input data file located at:', DATA_FILE_PATH)
print('\nDictionary preview:')
print(dictionary_preview)

Global random seed set to: 42
Input dictionary located at: data/participation_2024-25_data_dictionary_cleaned.txt
Input data file located at: data/participation_2024-25_experiment.tab

Dictionary preview:
UK Data Archive Data Dictionary

File-level information:

File Name = 		participation_2024-25_annual_data_open
Number of cases = 	34378


Variable-level information:

Pos. = 1,175	Variable = CARTS_NET	Variable label = In the last 12 months, engaged (attended OR participated) with the arts physically
This variable is    numeric, the SPSS measurement level is NOMINAL
SPSS user missing values = -1.7976931348623155e+308 thru -1.0
	Value label information for CARTS_NET
	Value = -3.0	Label = Not applicable
	Value = 1.0	Label = Yes
	Value = 2.0	Label = No
	Value = 3.0	Label = No & Missing

Pos. = 1,308	Variable = AGEBAND	Variable label = Respondent age band (ALL)



# 1 Dataset Ingestion and Problem Definition
## 1.1 Dataset Ingestion and Schema Checks
This subsection loads the experiment dataset into `participation_raw`, verifies expected variables and basic schema integrity, and records key ingestion diagnostics.

Checks performed in this subsection:
- Confirmed all required variables are present.
- Confirmed there are no duplicate column names.
- Checked that row count and column count are sensible for the provided file.
- Checked basic data types and whether unexpected nulls are present.
- Flagged any obvious schema anomalies relevant to ingestion (missing required fields, duplicate columns, null-heavy required variables).

In [2]:
required_vars = [
    'CARTS_NET', 'AGEBAND', 'SEX', 'QWORK', 'EDUCAT3',
    'FINHARD', 'CINTOFT', 'gor', 'rur11cat', 'CHILDHH', 'COHAB'
]

participation_raw = pd.read_csv(DATA_FILE_PATH, sep='	')
print('participation_raw shape:', participation_raw.shape)

missing_required = sorted(set(required_vars) - set(participation_raw.columns))
duplicate_columns = participation_raw.columns[participation_raw.columns.duplicated()].tolist()
null_counts = participation_raw[required_vars].isna().sum().sort_values(ascending=False)

dtype_summary = participation_raw[required_vars].dtypes.rename('dtype').reset_index().rename(columns={'index': 'variable'})
print('\nMissing required variables:', missing_required if missing_required else 'None')
print('Duplicate columns:', duplicate_columns if duplicate_columns else 'None')
print('\nNull counts in required variables:')
print(null_counts)
print('\nDtype summary:')
print(dtype_summary)

schema_issues = []
if missing_required:
    schema_issues.append(f'Missing required variables: {missing_required}')
if duplicate_columns:
    schema_issues.append(f'Duplicate columns: {duplicate_columns}')
if (null_counts > 0).any():
    schema_issues.append('Some required variables contain null values (coded missing may still be present as non-null codes).')

if schema_issues:
    print('\nSchema issues flagged:')
    for issue in schema_issues:
        print('-', issue)
else:
    print('\nNo blocking schema issues detected for this step.')

schema_report = dtype_summary.copy()
schema_report['null_count'] = schema_report['variable'].map(null_counts.to_dict())
schema_report.to_csv(EVIDENCE_DIR / 'schema_checks_codex.csv', index=False)

participation_raw shape: (34378, 11)

Missing required variables: None
Duplicate columns: None

Null counts in required variables:
CARTS_NET    0
AGEBAND      0
SEX          0
QWORK        0
EDUCAT3      0
FINHARD      0
CINTOFT      0
gor          0
rur11cat     0
CHILDHH      0
COHAB        0
dtype: int64

Dtype summary:
     variable  dtype
0   CARTS_NET  int64
1     AGEBAND  int64
2         SEX  int64
3       QWORK  int64
4     EDUCAT3  int64
5     FINHARD  int64
6     CINTOFT  int64
7         gor  int64
8    rur11cat  int64
9     CHILDHH  int64
10      COHAB  int64

No blocking schema issues detected for this step.


## 1.2 Prediction Task Definition
This project is a binary classification task: predict whether a respondent **did not engage physically with the arts** in the last 12 months.

We frame this as an under-engagement identification problem with social research value. Instead of treating arts participation as only an individual preference, the analysis assesses whether non-participation is patterned across demographic, socioeconomic, digital, and geographic factors, supporting evidence for more inclusive policy.

- **Target variable:** `CARTS_NET`
- **Feature variables:** `AGEBAND, SEX, QWORK, EDUCAT3, FINHARD, CINTOFT, gor, rur11cat, CHILDHH, COHAB`

For the target variable, rows where `CARTS_NET` is `-3` or `3` will be dropped later as missing target responses so the task becomes binary. This step does **not** drop them yet.

| Variable | Role | Meaning | Key coded values from dictionary |
|---|---|---|---|
| `CARTS_NET` | Target | Engaged physically with arts in last 12 months | `1=Yes`, `2=No`, `-3=Not applicable`, `3=No & Missing` |
| `AGEBAND` | Feature | Age band | `1=16-19` ... `15=85+`, `-3=Not applicable`, `997=Prefer not to say` |
| `SEX` | Feature | Respondent gender | `1=Female`, `2=Male`, negatives and `997` are non-informative |
| `QWORK` | Feature | Current working status | Categorical statuses `1,2,3,4,5,6,8,9,10`, with negatives/`997`/`999` non-informative |
| `EDUCAT3` | Feature | Highest qualification | `1=Degree+`, `2=Other qualification`, negatives/`997`/`999` non-informative |
| `FINHARD` | Feature | Financial management | `1=Living comfortably` to `5=Finding it very difficult`, negatives/`997` non-informative |
| `CINTOFT` | Feature | Internet use frequency | `1=Almost all the time` to `5=Less often`, negatives non-informative |
| `gor` | Feature | Region | `1` to `9` region categories |
| `rur11cat` | Feature | Rural/urban area | `1=Rural`, `2=Urban` |
| `CHILDHH` | Feature | Children in household | `0,1,2,3,4(4+)`, negatives/`997` non-informative |
| `COHAB` | Feature | Living as a couple | `1=Yes`, `2=No`, negatives/`997` non-informative |

# 2 Exploratory Data Analysis (EDA)
## 2.1 Target Cleaning for Binary Setup
Rows where `CARTS_NET` is coded as missing (`-3`, `3`) are removed. The original target column is then replaced with a new binary variable `under_engaged` (`1` = did not engage physically, `0` = engaged).

In [3]:
rows_before_target_clean = len(participation_raw)
participation_target_ready = participation_raw.loc[~participation_raw['CARTS_NET'].isin([-3, 3])].copy()
rows_after_target_clean = len(participation_target_ready)

participation_eda = participation_target_ready.drop(columns=['CARTS_NET']).copy()
participation_eda['under_engaged'] = (participation_target_ready['CARTS_NET'] == 2).astype(int)

print('Rows before target cleaning:', rows_before_target_clean)
print('Rows after target cleaning:', rows_after_target_clean)
print('Rows removed (target missing codes):', rows_before_target_clean - rows_after_target_clean)
print('\nunder_engaged distribution:')
print(participation_eda['under_engaged'].value_counts().sort_index())

target_dist = participation_eda['under_engaged'].value_counts(normalize=True).rename_axis('under_engaged').reset_index(name='proportion')
target_dist.to_csv(EVIDENCE_DIR / 'target_distribution_after_cleaning_codex.csv', index=False)

Rows before target cleaning: 34378
Rows after target cleaning: 34338
Rows removed (target missing codes): 40

under_engaged distribution:
under_engaged
0    31290
1     3048
Name: count, dtype: int64


## 2.2 Exploratory Visualisations
This EDA focuses on distributions of the target and features, feature-target relationships, and prevalence of coded non-informative responses.

In [4]:
sns.set_theme(style='whitegrid', context='notebook')

feature_cols = ['AGEBAND', 'SEX', 'QWORK', 'EDUCAT3', 'FINHARD', 'CINTOFT', 'gor', 'rur11cat', 'CHILDHH', 'COHAB']

def save_plot(fig, filename):
    out_path = EDA_PICS_DIR / filename
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)

# Plot 1: target distribution
fig, ax = plt.subplots(figsize=(6, 4))
vc = participation_eda['under_engaged'].value_counts().sort_index()
labels = ['Engaged (0)', 'Under-engaged (1)']
ax.bar(labels, vc.values, color=['#4c72b0', '#dd8452'])
ax.set_title('Target Distribution: Under-engaged vs Engaged')
ax.set_ylabel('Count')
save_plot(fig, '01_target_distribution.png')

# Plot 2: all feature distributions
fig, axes = plt.subplots(5, 2, figsize=(16, 22))
for ax, col in zip(axes.flatten(), feature_cols):
    order = sorted(participation_eda[col].dropna().unique())
    sns.countplot(data=participation_eda, x=col, order=order, ax=ax, color='#4c72b0')
    ax.set_title(f'Distribution of {col}')
    ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
save_plot(fig, '02_feature_distributions.png')

# Plot 3: key feature-target rates
selected_features = ['AGEBAND', 'FINHARD', 'CINTOFT', 'gor']
fig, axes = plt.subplots(2, 2, figsize=(16, 10))
for ax, col in zip(axes.flatten(), selected_features):
    rate = participation_eda.groupby(col)['under_engaged'].mean().reset_index().sort_values(col)
    sns.barplot(data=rate, x=col, y='under_engaged', ax=ax, color='#dd8452')
    ax.set_title(f'Under-engagement rate by {col}')
    ax.set_ylabel('Rate')
    ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
save_plot(fig, '03_under_engagement_rates_by_feature.png')

# Plot 4: coded non-informative response prevalence
coded_non_informative = {
    'AGEBAND': {-3, 997},
    'SEX': {-5, -4, -3, 997},
    'QWORK': {-5, -4, -3, 997, 999},
    'EDUCAT3': {-5, -4, -3, 997, 999},
    'FINHARD': {-5, -4, -3, 997},
    'CINTOFT': {-5, -4, -3},
    'gor': set(),
    'rur11cat': set(),
    'CHILDHH': {-6, -5, -4, -3, 997},
    'COHAB': {-5, -4, -3, 997},
}

missing_rates = []
for col in feature_cols:
    if coded_non_informative[col]:
        rate = participation_eda[col].isin(coded_non_informative[col]).mean()
    else:
        rate = 0.0
    missing_rates.append({'variable': col, 'coded_non_informative_rate': rate})

missing_rates_df = pd.DataFrame(missing_rates).sort_values('coded_non_informative_rate', ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(data=missing_rates_df, x='variable', y='coded_non_informative_rate', ax=ax, color='#55a868')
ax.set_title('Rate of Coded Non-informative Responses by Feature')
ax.set_ylabel('Rate')
ax.tick_params(axis='x', rotation=45)
fig.tight_layout()
save_plot(fig, '04_coded_non_informative_rates.png')

missing_rates_df.to_csv(EVIDENCE_DIR / 'coded_non_informative_rates_codex.csv', index=False)
print('Saved EDA figures to:', EDA_PICS_DIR)
print(missing_rates_df)

Saved EDA figures to: evidence_codex/EDA_codex_Pics
   variable  coded_non_informative_rate
9     COHAB                    0.711573
3   EDUCAT3                    0.239938
5   CINTOFT                    0.059089
4   FINHARD                    0.048110
2     QWORK                    0.039723
1       SEX                    0.018580
0   AGEBAND                    0.017648
8   CHILDHH                    0.002009
6       gor                    0.000000
7  rur11cat                    0.000000


# 3 Missingness Handling
Coded non-informative values are handled at the feature level using dictionary-based rules.

Handling rules used:
- Codes representing non-response, not applicable, and similar non-informative responses are mapped to an explicit category: `Unknown`.
- Informative response codes are retained and converted to readable categorical labels where possible.
- This approach preserves sample size while making missingness explicit for downstream models.

In [5]:
value_label_maps = {
    'AGEBAND': {
        1: '16-19', 2: '20-24', 3: '25-29', 4: '30-34', 5: '35-39',
        6: '40-44', 7: '45-49', 8: '50-54', 9: '55-59', 10: '60-64',
        11: '65-69', 12: '70-74', 13: '75-79', 14: '80-84', 15: '85+'
    },
    'SEX': {1: 'Female', 2: 'Male'},
    'QWORK': {
        1: 'Working full-time', 2: 'Working part-time', 3: 'Training/education',
        4: 'Unemployed', 5: 'Parental leave', 6: 'Retired', 8: 'Home/family',
        9: 'Sick/disabled', 10: 'Other'
    },
    'EDUCAT3': {1: 'Degree or above', 2: 'Other qualification'},
    'FINHARD': {
        1: 'Living comfortably', 2: 'Doing alright', 3: 'Just getting by',
        4: 'Finding quite difficult', 5: 'Finding very difficult'
    },
    'CINTOFT': {
        1: 'Almost all the time', 2: 'Many times a day', 3: 'About once a day',
        4: 'Several times a week', 5: 'Less often'
    },
    'gor': {
        1: 'North East', 2: 'North West', 3: 'Yorkshire and The Humber',
        4: 'East Midlands', 5: 'West Midlands', 6: 'East of England',
        7: 'London', 8: 'South East', 9: 'South West'
    },
    'rur11cat': {1: 'Rural', 2: 'Urban'},
    'CHILDHH': {0: '0', 1: '1', 2: '2', 3: '3', 4: '4+'},
    'COHAB': {1: 'Yes', 2: 'No'},
}

unknown_code_map = {
    'AGEBAND': {-3, 997},
    'SEX': {-5, -4, -3, 997},
    'QWORK': {-5, -4, -3, 997, 999},
    'EDUCAT3': {-5, -4, -3, 997, 999},
    'FINHARD': {-5, -4, -3, 997},
    'CINTOFT': {-5, -4, -3},
    'gor': set(),
    'rur11cat': set(),
    'CHILDHH': {-6, -5, -4, -3, 997},
    'COHAB': {-5, -4, -3, 997},
}

rows_before_missingness = len(participation_eda)
participation_clean = participation_eda.copy()
handling_records = []

for col in feature_cols:
    unknown_codes = unknown_code_map[col]
    labels = value_label_maps[col]

    unknown_count = participation_clean[col].isin(unknown_codes).sum() if unknown_codes else 0

    def transform_value(x):
        if pd.isna(x):
            return 'Unknown'
        if x in unknown_codes:
            return 'Unknown'
        return labels.get(x, f'Code_{int(x)}')

    participation_clean[col] = participation_clean[col].apply(transform_value)

    handling_records.append({
        'variable': col,
        'rows_mapped_to_unknown': int(unknown_count),
        'unknown_rate': float(unknown_count / rows_before_missingness),
    })

rows_after_missingness = len(participation_clean)
na_after = participation_clean.isna().sum().sum()

print('Rows before missingness handling:', rows_before_missingness)
print('Rows after missingness handling:', rows_after_missingness)
print('Total NaN values after handling:', int(na_after))

handling_summary_df = pd.DataFrame(handling_records).sort_values('unknown_rate', ascending=False)
print('\nMissingness handling summary:')
print(handling_summary_df)

assert na_after == 0, 'participation_clean still has missing values.'

handling_summary_df.to_csv(EVIDENCE_DIR / 'missingness_handling_summary_codex.csv', index=False)
participation_clean.to_csv(EVIDENCE_DIR / 'participation_clean_codex.csv', index=False)

Rows before missingness handling: 34338
Rows after missingness handling: 34338
Total NaN values after handling: 0

Missingness handling summary:
   variable  rows_mapped_to_unknown  unknown_rate
9     COHAB                   24434      0.711573
3   EDUCAT3                    8239      0.239938
5   CINTOFT                    2029      0.059089
4   FINHARD                    1652      0.048110
2     QWORK                    1364      0.039723
1       SEX                     638      0.018580
0   AGEBAND                     606      0.017648
8   CHILDHH                      69      0.002009
6       gor                       0      0.000000
7  rur11cat                       0      0.000000


# 4 Modeling Preparation and Evaluation Harness
## 4.1 Prepare Modeling Data
Define `X` and `y`, build preprocessing pipelines for Logistic Regression and XGBoost, and split data into train/validation/test with stratification (0.7 / 0.15 / 0.15).

In [6]:
X = participation_clean[feature_cols].copy()
y = participation_clean['under_engaged'].astype(int).copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=y_temp,
)

# Compatible OneHotEncoder setup across sklearn versions
try:
    ohe_lr = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
    ohe_xgb = OneHotEncoder(handle_unknown='ignore', sparse_output=True)
except TypeError:
    ohe_lr = OneHotEncoder(handle_unknown='ignore', sparse=True)
    ohe_xgb = OneHotEncoder(handle_unknown='ignore', sparse=True)

lr_preprocessor = ColumnTransformer(
    transformers=[('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', ohe_lr)]), feature_cols)],
    remainder='drop',
)

xgb_preprocessor = ColumnTransformer(
    transformers=[('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('ohe', ohe_xgb)]), feature_cols)],
    remainder='drop',
)

split_summary = pd.DataFrame([
    {'split': 'train', 'rows': len(X_train), 'positive_rate': y_train.mean()},
    {'split': 'validation', 'rows': len(X_val), 'positive_rate': y_val.mean()},
    {'split': 'test', 'rows': len(X_test), 'positive_rate': y_test.mean()},
])

print(split_summary)
split_summary.to_csv(EVIDENCE_DIR / 'split_summary_codex.csv', index=False)

        split   rows  positive_rate
0       train  24036       0.088783
1  validation   5151       0.088721
2        test   5151       0.088721


## 4.2 Common Evaluation Harness
The same metrics are used for both Logistic Regression and XGBoost to ensure fair comparison.

Design rationale:
- The policy use case prioritizes identifying under-engaged people, so **recall** for the positive class is primary.
- **Precision** is tracked to monitor false-positive burden.
- **F1** captures balance between precision and recall.
- **PR-AUC** is included because the positive class is relatively rare.
- **ROC-AUC** and **balanced accuracy** provide additional discrimination and class-balance diagnostics.
- **Confusion matrix** supports transparent error interpretation.

In [7]:
def evaluate_model(estimator, X_eval, y_eval, model_name, split_name, threshold=0.5):
    prob = estimator.predict_proba(X_eval)[:, 1]
    pred = (prob >= threshold).astype(int)

    metrics = {
        'model': model_name,
        'split': split_name,
        'threshold': threshold,
        'recall': recall_score(y_eval, pred),
        'precision': precision_score(y_eval, pred, zero_division=0),
        'f1': f1_score(y_eval, pred, zero_division=0),
        'pr_auc': average_precision_score(y_eval, prob),
        'roc_auc': roc_auc_score(y_eval, prob),
        'balanced_accuracy': balanced_accuracy_score(y_eval, pred),
    }

    tn, fp, fn, tp = confusion_matrix(y_eval, pred).ravel()
    metrics.update({'tn': int(tn), 'fp': int(fp), 'fn': int(fn), 'tp': int(tp)})
    return metrics


def show_metrics(metrics_dict):
    display_cols = ['model', 'split', 'recall', 'precision', 'f1', 'pr_auc', 'roc_auc', 'balanced_accuracy', 'tn', 'fp', 'fn', 'tp']
    return pd.DataFrame([metrics_dict])[display_cols]

## 4.3 Baseline Logistic Regression
Train a baseline Logistic Regression model with standard, reasonable settings and evaluate on the validation set only.

In [8]:
baseline_lr_pipeline = Pipeline([
    ('preprocess', lr_preprocessor),
    ('model', LogisticRegression(
        random_state=RANDOM_STATE,
        class_weight='balanced',
        max_iter=3000,
        solver='liblinear',
    )),
])

baseline_lr_pipeline.fit(X_train, y_train)
baseline_lr_val_metrics = evaluate_model(baseline_lr_pipeline, X_val, y_val, 'baseline_logistic_regression', 'validation')

show_metrics(baseline_lr_val_metrics)

pd.DataFrame([baseline_lr_val_metrics]).to_csv(EVIDENCE_DIR / 'baseline_lr_validation_metrics_codex.csv', index=False)

# 5 Model Improvement, Comparison, and Final Decision
## 5.1 Tune Logistic Regression
Use the same train/validation/test split and tune Logistic Regression on the validation set only.

In [9]:
lr_param_grid = {
    'C': [0.01, 0.1, 1.0, 3.0, 10.0],
    'penalty': ['l1', 'l2'],
}

lr_trials = []
best_lr_tuple = (-1, -1, -1, -1)
best_lr_model = None
best_lr_params = None
best_lr_metrics = None

for trial_idx, params in enumerate(ParameterGrid(lr_param_grid), start=1):
    candidate = Pipeline([
        ('preprocess', lr_preprocessor),
        ('model', LogisticRegression(
            random_state=RANDOM_STATE,
            class_weight='balanced',
            max_iter=3000,
            solver='liblinear',
            C=params['C'],
            penalty=params['penalty'],
        )),
    ])

    candidate.fit(X_train, y_train)
    metrics = evaluate_model(candidate, X_val, y_val, 'tuned_logistic_regression', 'validation')

    trial_record = {
        'trial': trial_idx,
        'C': params['C'],
        'penalty': params['penalty'],
        **{k: metrics[k] for k in ['recall', 'precision', 'f1', 'pr_auc', 'roc_auc', 'balanced_accuracy']},
    }
    lr_trials.append(trial_record)

    score_tuple = (metrics['recall'], metrics['f1'], metrics['pr_auc'], metrics['balanced_accuracy'])
    if score_tuple > best_lr_tuple:
        best_lr_tuple = score_tuple
        best_lr_model = candidate
        best_lr_params = params
        best_lr_metrics = metrics

lr_trials_df = pd.DataFrame(lr_trials).sort_values(['recall', 'f1', 'pr_auc', 'balanced_accuracy'], ascending=False)
lr_trials_df.to_csv(EVIDENCE_DIR / 'lr_tuning_trials_codex.csv', index=False)

tuned_lr_pipeline = best_lr_model
print('Best Logistic Regression params:', best_lr_params)
show_metrics(best_lr_metrics)

Best Logistic Regression params: {'C': 1.0, 'penalty': 'l1'}


,model,split,recall,precision,f1,pr_auc,roc_auc,balanced_accuracy,tn,fp,fn,tp
0,tuned_logistic_regression,validation,0.652079,0.194771,0.29995,0.250706,0.763359,0.694808,3462,1232,159,298


## 5.2 Tune XGBoost
Use the same train/validation/test split and tune XGBoost on the validation set only.

In [10]:
neg_count = int((y_train == 0).sum())
pos_count = int((y_train == 1).sum())
scale_pos_weight = neg_count / pos_count

xgb_param_grid = {
    'n_estimators': [200, 350],
    'max_depth': [3, 5],
    'learning_rate': [0.05, 0.1],
    'colsample_bytree': [0.8, 1.0],
}

xgb_trials = []
best_xgb_tuple = (-1, -1, -1, -1)
best_xgb_model = None
best_xgb_params = None
best_xgb_metrics = None

for trial_idx, params in enumerate(ParameterGrid(xgb_param_grid), start=1):
    xgb_clf = XGBClassifier(
        objective='binary:logistic',
        eval_metric='logloss',
        random_state=RANDOM_STATE,
        n_jobs=-1,
        tree_method='hist',
        subsample=0.8,
        scale_pos_weight=scale_pos_weight,
        reg_lambda=1.0,
        min_child_weight=1,
        **params,
    )

    candidate = Pipeline([
        ('preprocess', xgb_preprocessor),
        ('model', xgb_clf),
    ])

    candidate.fit(X_train, y_train)
    metrics = evaluate_model(candidate, X_val, y_val, 'tuned_xgboost', 'validation')

    trial_record = {
        'trial': trial_idx,
        **params,
        **{k: metrics[k] for k in ['recall', 'precision', 'f1', 'pr_auc', 'roc_auc', 'balanced_accuracy']},
    }
    xgb_trials.append(trial_record)

    score_tuple = (metrics['recall'], metrics['f1'], metrics['pr_auc'], metrics['balanced_accuracy'])
    if score_tuple > best_xgb_tuple:
        best_xgb_tuple = score_tuple
        best_xgb_model = candidate
        best_xgb_params = params
        best_xgb_metrics = metrics

xgb_trials_df = pd.DataFrame(xgb_trials).sort_values(['recall', 'f1', 'pr_auc', 'balanced_accuracy'], ascending=False)
xgb_trials_df.to_csv(EVIDENCE_DIR / 'xgb_tuning_trials_codex.csv', index=False)

tuned_xgb_pipeline = best_xgb_model
print('Best XGBoost params:', best_xgb_params)
show_metrics(best_xgb_metrics)

Best XGBoost params: {'colsample_bytree': 1.0, 'learning_rate': 0.05, 'max_depth': 3, 'n_estimators': 200}


,model,split,recall,precision,f1,pr_auc,roc_auc,balanced_accuracy,tn,fp,fn,tp
0,tuned_xgboost,validation,0.682713,0.201942,0.311688,0.252596,0.768035,0.710019,3461,1233,145,312


## 5.3 Model Comparison on Test Set
Only at this stage, all candidate models are evaluated on the held-out test set.

In [11]:
baseline_test = evaluate_model(baseline_lr_pipeline, X_test, y_test, 'baseline_logistic_regression', 'test')
tuned_lr_test = evaluate_model(tuned_lr_pipeline, X_test, y_test, 'tuned_logistic_regression', 'test')
tuned_xgb_test = evaluate_model(tuned_xgb_pipeline, X_test, y_test, 'tuned_xgboost', 'test')

comparison_df = pd.DataFrame([baseline_test, tuned_lr_test, tuned_xgb_test])
comparison_df = comparison_df[['model', 'split', 'recall', 'precision', 'f1', 'pr_auc', 'roc_auc', 'balanced_accuracy', 'tn', 'fp', 'fn', 'tp']]
comparison_df.to_csv(EVIDENCE_DIR / 'model_test_comparison_codex.csv', index=False)
print(comparison_df)

fig, ax = plt.subplots(figsize=(8, 5))
plot_df = comparison_df[['model', 'recall', 'f1', 'pr_auc']].melt(id_vars='model', var_name='metric', value_name='value')
sns.barplot(data=plot_df, x='metric', y='value', hue='model', ax=ax)
ax.set_title('Test-set comparison across key metrics')
ax.set_ylim(0, 1)
fig.tight_layout()
fig.savefig(EVIDENCE_DIR / 'test_metric_comparison_codex.png', dpi=150, bbox_inches='tight')
plt.close(fig)

                          model split    recall  precision        f1  \
0  baseline_logistic_regression  test  0.678337   0.211316  0.322245   
1     tuned_logistic_regression  test  0.676149   0.210922  0.321540   
2                 tuned_xgboost  test  0.689278   0.208058  0.319635   

     pr_auc   roc_auc  balanced_accuracy    tn    fp   fn   tp  
0  0.288423  0.786014           0.715926  3537  1157  147  310  
1  0.288480  0.786095           0.714938  3538  1156  148  309  
2  0.292827  0.789739           0.716923  3495  1199  142  315  


## 5.4 Final Model Decision Framework
Selection framework (quantitative, multi-dimensional):
- Primary criterion: **Recall** for under-engaged respondents (weight 0.50).
- Secondary criteria: **F1** (0.20), **PR-AUC** (0.15), **Balanced Accuracy** (0.10), **Precision** (0.05).
- Models compared for final choice: tuned Logistic Regression and tuned XGBoost.

This weighting emphasizes missed under-engaged respondents as the highest policy risk while still accounting for false positives and overall discrimination.

In [12]:
weights = {
    'recall': 0.50,
    'f1': 0.20,
    'pr_auc': 0.15,
    'balanced_accuracy': 0.10,
    'precision': 0.05,
}

final_candidates = comparison_df[comparison_df['model'].isin(['tuned_logistic_regression', 'tuned_xgboost'])].copy()

for metric, w in weights.items():
    final_candidates[f'w_{metric}'] = final_candidates[metric] * w
final_candidates['selection_score'] = final_candidates[[f'w_{m}' for m in weights]].sum(axis=1)

final_candidates = final_candidates.sort_values('selection_score', ascending=False)
final_model_name = final_candidates.iloc[0]['model']

lr_tuning_summary = {
    'model': 'tuned_logistic_regression',
    'tuning_method_used': 'Manual grid search on validation set',
    'hyperparameters_searched': ['C', 'penalty'],
    'search_range': {'C': [0.01, 0.1, 1.0, 3.0, 10.0], 'penalty': ['l1', 'l2']},
    'total_parameter_configurations_evaluated': int(len(lr_trials_df)),
    'iteration_or_trial_count_completed': int(len(lr_trials_df)),
    'best_hyperparameter_setting_found': best_lr_params,
    'best_validation_set_performance': {k: float(best_lr_metrics[k]) for k in ['recall', 'precision', 'f1', 'pr_auc', 'roc_auc', 'balanced_accuracy']},
}

xgb_tuning_summary = {
    'model': 'tuned_xgboost',
    'tuning_method_used': 'Manual grid search on validation set',
    'hyperparameters_searched': ['n_estimators', 'max_depth', 'learning_rate', 'colsample_bytree'],
    'search_range': {'n_estimators': [200, 350], 'max_depth': [3, 5], 'learning_rate': [0.05, 0.1], 'colsample_bytree': [0.8, 1.0], 'subsample': [0.8]},
    'total_parameter_configurations_evaluated': int(len(xgb_trials_df)),
    'iteration_or_trial_count_completed': int(len(xgb_trials_df)),
    'best_hyperparameter_setting_found': best_xgb_params,
    'best_validation_set_performance': {k: float(best_xgb_metrics[k]) for k in ['recall', 'precision', 'f1', 'pr_auc', 'roc_auc', 'balanced_accuracy']},
}

print('Structured tuning summary: Logistic Regression')
print(json.dumps(lr_tuning_summary, indent=2))
print('\nStructured tuning summary: XGBoost')
print(json.dumps(xgb_tuning_summary, indent=2))

print('\nFinal candidate scores:')
print(final_candidates[['model', 'selection_score', 'recall', 'precision', 'f1', 'pr_auc', 'roc_auc', 'balanced_accuracy']])
print('\nSelected final model:', final_model_name)

with (EVIDENCE_DIR / 'tuning_summary_codex.json').open('w', encoding='utf-8') as f:
    json.dump({'logistic_regression': lr_tuning_summary, 'xgboost': xgb_tuning_summary}, f, indent=2)

final_candidates.to_csv(EVIDENCE_DIR / 'final_model_decision_scores_codex.csv', index=False)

Structured tuning summary: Logistic Regression
{
  "model": "tuned_logistic_regression",
  "tuning_method_used": "Manual grid search on validation set",
  "hyperparameters_searched": [
    "C",
    "penalty"
  ],
  "search_range": {
    "C": [
      0.01,
      0.1,
      1.0,
      3.0,
      10.0
    ],
    "penalty": [
      "l1",
      "l2"
    ]
  },
  "total_parameter_configurations_evaluated": 10,
  "iteration_or_trial_count_completed": 10,
  "best_hyperparameter_setting_found": {
    "C": 1.0,
    "penalty": "l1"
  },
  "best_validation_set_performance": {
    "recall": 0.6520787746170679,
    "precision": 0.19477124183006536,
    "f1": 0.2999496728736789,
    "pr_auc": 0.2507061808162201,
    "roc_auc": 0.7633591558290811,
    "balanced_accuracy": 0.6948080281265996
  }
}

Structured tuning summary: XGBoost
{
  "model": "tuned_xgboost",
  "tuning_method_used": "Manual grid search on validation set",
  "hyperparameters_searched": [
    "n_estimators",
    "max_depth",
    "lear